In [1]:
import numpy as np
import pandas as pd


In [3]:
df=pd.read_csv("diabetes.csv")
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [7]:
X = df.iloc[:, :-1].values  # Convert to NumPy array
print(X[:5])
y = df.iloc[:, -1].values   # The last column (target)
print(y[:5])

[[6.000e+00 1.480e+02 7.200e+01 3.500e+01 0.000e+00 3.360e+01 6.270e-01
  5.000e+01]
 [1.000e+00 8.500e+01 6.600e+01 2.900e+01 0.000e+00 2.660e+01 3.510e-01
  3.100e+01]
 [8.000e+00 1.830e+02 6.400e+01 0.000e+00 0.000e+00 2.330e+01 6.720e-01
  3.200e+01]
 [1.000e+00 8.900e+01 6.600e+01 2.300e+01 9.400e+01 2.810e+01 1.670e-01
  2.100e+01]
 [0.000e+00 1.370e+02 4.000e+01 3.500e+01 1.680e+02 4.310e+01 2.288e+00
  3.300e+01]]
[1 0 1 0 1]


In [9]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier

# Initialize model
model = RandomForestClassifier()

# Perform Sequential Forward Selection (SFS)
sfs = SequentialFeatureSelector(model, n_features_to_select=5, direction="forward")
sfs.fit(X, y)

# Get selected features
cols=df.columns[0:len(df.columns)-1]
selected_features_sfs = cols[sfs.support_].tolist() #sfs.support_: Returns True for selected features, False for ignored ones.
print("Top 5 Features using SFS:", selected_features_sfs)

Top 5 Features using SFS: ['Pregnancies', 'Glucose', 'BloodPressure', 'BMI', 'Age']


In [11]:
x=df[['Pregnancies', 'Glucose', 'BloodPressure', 'BMI', 'Age']]

In [13]:
x

,Pregnancies,Glucose,BloodPressure,BMI,Age
0,6,148,72,33.6,50
1,1,85,66,26.6,31
2,8,183,64,23.3,32
3,1,89,66,28.1,21
4,0,137,40,43.1,33
...,...,...,...,...,...
763,10,101,76,32.9,63
764,2,122,70,36.8,27
765,5,121,72,26.2,30
766,1,126,60,30.1,47


In [21]:
y=df['Outcome']

In [23]:
y

0      1
1      0
2      1
3      0
4      1
      ..
763    0
764    0
765    0
766    1
767    0
Name: Outcome, Length: 768, dtype: int64

In [57]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,recall_score,precision_score,f1_score

In [35]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [37]:
model=RandomForestClassifier()

In [41]:
model.fit(x_train,y_train)

RandomForestClassifier()

In [53]:
y_pred=model.predict(x_test)

In [59]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro")  # Macro average for multi-class
recall = recall_score(y_test, y_pred,average="macro")
f1 = f1_score(y_test, y_pred, average="macro")

# Print the results
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

Accuracy: 0.75
Precision: 0.73
Recall: 0.74
F1 Score: 0.73


In [69]:
from sklearn.model_selection import RandomizedSearchCV, train_test_split
grid={"n_estimators":[10,100,200,500,1000,1200],
      "max_depth":[None,5,10,20,30],
      'max_features': ['sqrt', 'log2', None],
      "min_samples_split":[2,4,6],
      "min_samples_leaf":[1,2,4]}
np.random.seed(42)
x=df[['Pregnancies', 'Glucose', 'BloodPressure', 'BMI', 'Age']]
y=df["Outcome"]
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)
model=RandomForestClassifier(n_jobs=1)
rs_model=RandomizedSearchCV(estimator=model,
                            param_distributions=grid,
                            n_iter=10,
                            cv=5,
                            verbose=2)
rs_model.fit(x_train,y_train);


Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=6, n_estimators=200; total time=   0.5s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=6, n_estimators=200; total time=   0.5s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=6, n_estimators=200; total time=   0.4s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=6, n_estimators=200; total time=   0.3s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=6, n_estimators=200; total time=   0.3s
[CV] END max_depth=5, max_features=None, min_samples_leaf=4, min_samples_split=2, n_estimators=500; total time=   1.1s
[CV] END max_depth=5, max_features=None, min_samples_leaf=4, min_samples_split=2, n_estimators=500; total time=   1.1s
[CV] END max_depth=5, max_features=None, min_samples_leaf=4, min_samples_split=2, n_estimators=500; t

In [75]:
rs_model.score(x_test,y_test)

0.7532467532467533

In [79]:
import pickle
with open("diabetes.pkl","wb")as file:
    pickle.dump(rs_model,file)
print("sucess")

sucess
